# 弗拉基米尔·普京维基百科摘要示例

## 练习目标（理念）

演示如何用 **OpenAI Chat Completions**，把「抓取到的维基百科正文」整理成**事实、中立、结构化**的人物摘要：

- **输入**：`https://en.wikipedia.org/wiki/Vladimir_Putin`（经本地 `scraper.fetch_website_contents`）
- **结构**：`system_prompt`（角色与大纲）+ `user_prompt`（任务 + 正文）→ `messages` 列表
- **输出**：模型返回 Markdown，在笔记本里 `display`

这是 Day 1「网页 → 提示词 → API → 展示」流水线的一次完整演练；主题是政治人物维基，重点仍是 **prompt 分层** 与 **messages 组装**。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取封装 | `from scraper import fetch_website_contents` |
| System / User 提示 | 政治分析师角色 vs 具体摘要任务 |
| `messages` 列表 | `[{"role":"system",...},{"role":"user",...}]` |
| Chat Completions | `openai.chat.completions.create(model="gpt-4.1-mini", ...)` |
| 笔记本展示 | `display(Markdown(result))` |

## 怎么跑

1. 准备好 `.env` 中的 OpenAI 密钥（`OPENAI_API_KEY` 等，以你环境为准）
2. 确保同目录（或 PYTHONPATH）能导入 `scraper.py` 里的 `fetch_website_contents`
3. 从上到下运行：导入客户端 → 写 system → 抓维基并写 user → 组 messages → 调用并展示


In [ ]:
# ========== 导入与客户端：环境、抓取工具、OpenAI ==========

# 导入标准库 os：虽本格未直接用，常与环境变量搭配保留
import os
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量，避免写死在代码里
from dotenv import load_dotenv
# 从本地 scraper 模块导入 fetch_website_contents：封装好的网页正文抓取函数
from scraper import fetch_website_contents
# 从 IPython.display 导入 Markdown、display：把模型输出渲染成笔记本 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI：创建官方云端客户端
from openai import OpenAI

# override=True：用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 无参构造：默认从环境变量读取 OPENAI_API_KEY，并指向官方 API
openai = OpenAI()


In [ ]:
# ========== 第 1 步：system_prompt —— 角色、中立立场与输出大纲 ==========

# 三引号多行字符串：告诉模型「你是谁、覆盖哪些要点、用什么格式」
# 内容保持英文（可运行 / 影响回答的 prompt 不翻译）
system_prompt = """
You are a concise political analyst. When given information about a world leader,
provide a factual, neutral, and well-structured summary covering:
- Early life and rise to power
- Key political positions and ideology
- Major domestic and foreign policy decisions
- International controversies and legacy
Respond in markdown. Do not wrap the markdown in a code block.
"""


In [ ]:
# ========== 第 2 步：user_prompt —— 抓取维基正文并拼进任务说明 ==========

# 调用本地抓取函数；URL 保持英文维基地址（影响抓取目标，不翻译）
putin_wiki = fetch_website_contents("https://en.wikipedia.org/wiki/Vladimir_Putin")

# 用括号拼接多段字符串：先写任务说明，再把整页正文接在后面
user_prompt = (
    "Below is the Wikipedia content about Vladimir Putin.\n"
    "Please provide a concise, structured summary of his life, "
    "political career, and legacy.\n\n"
    + putin_wiki
)


In [ ]:
# ========== 第 3 步：messages —— API 期望的 role/content 列表 ==========

# Chat Completions 约定：列表里每项是一条消息；通常 system 在前、user 在后
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": user_prompt}
]


In [ ]:
# ========== 第 4 步：调用 API 并展示 Markdown 结果 ==========

# 非流式 create：传入模型 id 与完整 messages；返回 Completions 响应对象
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
)

# 从 choices[0].message.content 取出助手生成的正文
result = response.choices[0].message.content
# 在笔记本中按 Markdown 渲染（比 print 更易读标题/列表）
display(Markdown(result))
